In [1]:
from datasets import load_dataset

ds = load_dataset("Teklia/IAM-line")
print(ds)
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/validation.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 6482
    })
    validation: Dataset({
        features: ['image', 'text'],
        num_rows: 976
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 2915
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=L size=2467x128 at 0x7F2E0ADE0D70>, 'text': 'put down a resolution on the subject'}


In [2]:
import string

all_text = ""
for sample in ds["train"]:
    all_text += sample["text"]

vocab = sorted(list(set(all_text)))

vocab.append("<blank>")

char2idx = {char: idx for idx, char in enumerate(vocab)}
idx2char = {idx: char for char, idx in char2idx.items()}

print("Vocab size:", len(vocab))

Vocab size: 80


In [3]:
def encode_text(text):
    return [char2idx[c] for c in text]

In [4]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [5]:
from torch.utils.data import Dataset

class IAMDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = transform(sample["image"])
        label = encode_text(sample["text"])
        return image, label

In [6]:
from torch.utils.data import DataLoader

train_dataset = IAMDataset(ds["train"])

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=lambda x: x
)

In [7]:
import torch
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128,256,3,padding=1),
            nn.ReLU(),
        )

        self.rnn = nn.LSTM(
            input_size=256 * 8,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()

        x = x.permute(0, 3, 1, 2)
        x = x.view(b, w, c * h)

        x, _ = self.rnn(x)
        x = self.fc(x)

        return x

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN(num_classes=len(vocab)).to(device)

criterion = nn.CTCLoss(
    blank=char2idx["<blank>"],
    zero_infinity=True
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

In [9]:
def decode_predictions(outputs, idx2char, blank_idx):
    preds = torch.argmax(outputs, dim=2)
    decoded = []

    for pred in preds:
        prev = None
        text = ""

        for p in pred:
            p = p.item()
            if p != blank_idx and p != prev:
                text += idx2char[p]
            prev = p

        decoded.append(text)

    return decoded

In [10]:
print(model is not None)
print(transform is not None)
print(idx2char is not None)

True
True
True


In [184]:
def predict_image(pil_img):
    model.eval()

    # transform (same as training)
    img = transform(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        outputs = outputs.log_softmax(2)

    preds = decode_predictions(
        outputs,
        idx2char,
        char2idx["<blank>"]
    )

    return preds[0]

In [ ]:
MAX_LABEL_LENGTH = 40

for epoch in range(30):
    model.train()

    total_loss = 0

    for batch in train_loader:
        images = []
        labels = []
        target_lengths = []

        # ---- prepare batch ----
        for img, lbl in batch:
            lbl = lbl[:MAX_LABEL_LENGTH]
            images.append(img)
            labels.append(torch.tensor(lbl, dtype=torch.long))
            target_lengths.append(len(lbl))

        images = torch.stack(images).to(device)
        labels = torch.cat(labels).to(device)
        target_lengths = torch.tensor(target_lengths, dtype=torch.long).to(device)

        # ---- forward ----
        outputs = model(images)
        outputs = outputs.log_softmax(2)
        outputs = outputs.permute(1, 0, 2)

        input_lengths = torch.full(
            (images.size(0),),
            outputs.size(0),
            dtype=torch.long
        ).to(device)

        # ---- loss ----
        loss = criterion(
            outputs,
            labels,
            input_lengths,
            target_lengths
        )

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ---- evaluation ----
    if epoch % 2 == 0:
        model.eval()

        with torch.no_grad():
            sample_batch = next(iter(train_loader))

            images = []
            texts = []

            for img, lbl in sample_batch:
                lbl = lbl[:MAX_LABEL_LENGTH]
                images.append(img)
                texts.append(lbl)

            images = torch.stack(images).to(device)

            outputs = model(images)
            outputs = outputs.log_softmax(2)

            preds = decode_predictions(
                outputs,
                idx2char,
                char2idx["<blank>"]
            )

            print(f"\n--- Epoch {epoch} Predictions ---")

            for i in range(min(5, len(preds))):
                true_text = "".join([idx2char[c] for c in texts[i]])
                print(f"GT: {true_text}")
                print(f"PR: {preds[i]}")
                print()

    print(f"Epoch {epoch}, Avg Loss: {avg_loss:.4f}")


--- Epoch 0 Predictions ---
GT: It has aroused strong opposition from th
PR: It has orowsed sfrong oppoition froun t

GT: girl lives in a single dingy room with h
PR: girl lives in a single dingy rom with 

GT: Market's Council of Ministers draws up t
PR: Market's Council of Ministers daws up the

GT: take ruthless action against the drug ma
PR: take ruthess action agaist the derng m

GT: Cambridge want the study of agriculture 
PR: Cambrvidge wast the study of agricultis 

Epoch 0, Avg Loss: 0.4938
Epoch 1, Avg Loss: 0.4126

--- Epoch 2 Predictions ---
GT: the alternatives had been put clearly to
PR: the altermatives had been put clearty to

GT: allied . But governments should be free 
PR: allied . But govenments hould be frr

GT: family man or father-to-be . Unlike many
PR: family man or father-to-be . Ulitle ona

GT: threat to those who toil and spin has be
PR: threat to those who toil and spin has be

GT: principal Nato ally " grows stronger
PR: principal Nato ally " grows strouge

In [ ]:
torch.save(model.state_dict(), "crnn_iam.pth")

Fine Tuning OCR Model on Medical Data

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
BASE_PATH = "/content/drive/MyDrive/Doctor’s Handwritten Prescription BD dataset"

In [14]:
import pandas as pd

train_df = pd.read_csv(f"{BASE_PATH}/training/training_labels.csv")
train_df.head()

,IMAGE,MEDICINE_NAME,GENERIC_NAME
0,0.png,Aceta,Paracetamol
1,1.png,Aceta,Paracetamol
2,2.png,Aceta,Paracetamol
3,3.png,Aceta,Paracetamol
4,4.png,Aceta,Paracetamol


changing vocab catered for fine tuning

In [15]:
import string

vocab = list(string.ascii_lowercase)
vocab.append("<blank>")

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

print("Vocab size:", len(vocab))

Vocab size: 27


In [16]:
model.fc = nn.Linear(512, len(vocab)).to(device)

print(model.fc)

Linear(in_features=512, out_features=27, bias=True)


In [17]:
criterion = nn.CTCLoss(
    blank=char2idx["<blank>"],
    zero_infinity=True
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [18]:
from torch.utils.data import Dataset
from PIL import Image

class PrescriptionDataset(Dataset):
    def __init__(self, df, img_folder):
        self.df = df
        self.img_folder = img_folder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = f"{self.img_folder}/{row['IMAGE']}"
        image = Image.open(img_path).convert("L")

        image = transform(image)

        import re
        text = row["MEDICINE_NAME"].lower()
        text = re.sub(r"[^a-z]", "", text)

        if len(text) == 0:
          text = "a"
        label = encode_text(text)

        return image, label

In [19]:
train_dataset = PrescriptionDataset(
    train_df,
    f"{BASE_PATH}/training/training_words"
)

In [20]:
img, lbl = train_dataset[0]

print("Image shape:", img.shape)
print("Label:", lbl)

Image shape: torch.Size([1, 32, 256])
Label: [0, 2, 4, 19, 0]


In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda x: x,
    num_workers=2
)

just reloading

In [36]:
import json

with open("/content/drive/MyDrive/vocab.json", "r") as f:
    char2idx = json.load(f)

# IMPORTANT: keys might be strings → convert
char2idx = {k: int(v) for k, v in char2idx.items()}

idx2char = {v: k for k, v in char2idx.items()}

In [37]:
model = CRNN(num_classes=len(char2idx)).to(device)

In [38]:
model.load_state_dict(
    torch.load("/content/drive/MyDrive/crnn_prescription.pth", map_location=device)
)

model.eval()

CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
  )
  (rnn): LSTM(2048, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=512, out_features=27, bias=True)
)

In [39]:
print(predict_image("265.png"))

exorax


back at it

In [34]:
MAX_LABEL_LENGTH = 15

for epoch in range(20):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        images = []
        labels = []
        target_lengths = []

        for img, lbl in batch:
            lbl = lbl[:MAX_LABEL_LENGTH]
            images.append(img)
            labels.append(torch.tensor(lbl, dtype=torch.long))
            target_lengths.append(len(lbl))

        images = torch.stack(images).to(device)
        labels = torch.cat(labels).to(device)
        target_lengths = torch.tensor(target_lengths).to(device)

        outputs = model(images)
        outputs = outputs.log_softmax(2)
        outputs = outputs.permute(1, 0, 2)

        input_lengths = torch.full(
            (images.size(0),),
            outputs.size(0),
            dtype=torch.long
        ).to(device)

        if torch.any(target_lengths > input_lengths):
            continue

        loss = criterion(outputs, labels, input_lengths, target_lengths)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch} | Batch {i} | Loss {loss.item():.4f}")

    print(f"\nEpoch {epoch}, Avg Loss: {total_loss / len(train_loader):.4f}\n")
    raw = predict_image("265.png")
    print("Sample:", raw)

Epoch 0 | Batch 0 | Loss 32.0862


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/crnn_prescription.pth")

In [23]:
import json

with open("/content/drive/MyDrive/vocab.json", "w") as f:
    json.dump(char2idx, f)

In [24]:
# recreate model
model2 = CRNN(num_classes=len(char2idx)).to(device)

# load weights
model2.load_state_dict(torch.load("/content/drive/MyDrive/crnn_prescription.pth"))
model2.eval()

print("Model loaded successfully")

Model loaded successfully


NLP CORRECTION PIPELINE

In [25]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("DereAbdulhameed/drug_names_dataset")

df = pd.DataFrame(ds["train"])
drug_list = df["drug_name"].str.lower().tolist()

train_drug_names.csv: 0.00B [00:00, ?B/s]

test_drug_names.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/506 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/127 [00:00<?, ? examples/s]

In [26]:
import re

def clean_drug(name):
    name = name.lower()
    name = re.sub(r"[^a-z]", "", name)
    return name

drug_list = list(set([clean_drug(d) for d in drug_list]))

In [27]:
train_drugs = train_df["MEDICINE_NAME"].str.lower().tolist()
train_drugs = [clean_drug(d) for d in train_drugs]

drug_list = list(set(drug_list + train_drugs))

print("Total drugs:", len(drug_list))

Total drugs: 569


In [28]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.6 MB/s eta 0:00:00


In [29]:
from rapidfuzz import process, fuzz

def correct_text(word):
    word = word.lower()

    matches = process.extract(
        word,
        drug_list,
        scorer=fuzz.token_sort_ratio,
        limit=3
    )

    best_match, score, _ = matches[0]

    if score > 65:
        return best_match

    return word

In [30]:
def predict_and_correct(image_path):
    raw = predict_image(image_path)
    corrected = correct_text(raw)

    return raw, corrected

In [40]:
raw, final = predict_and_correct("265.png")

print("OCR:", raw)
print("Corrected:", final)

OCR: exorax
Corrected: esoral


Mapping to Brand Names + Condidence Score

In [41]:
name_map = dict(zip(
    train_df["MEDICINE_NAME"].str.lower(),
    train_df["GENERIC_NAME"].str.lower()
))

In [42]:
import re

def clean_text_fn(x):
    return re.sub(r"[^a-z]", "", x.lower())

name_map = {
    clean_text_fn(k): v.lower()
    for k, v in zip(
        train_df["MEDICINE_NAME"],
        train_df["GENERIC_NAME"]
    )
}

In [43]:
train_drugs = list(set(train_drugs))

In [44]:
from rapidfuzz import process, fuzz

def correct_text_with_confidence(word):
    word = word.lower()

    candidates = process.extract(
        word,
        train_drugs,
        scorer=fuzz.WRatio,
        limit=5
    )

    best_match = word
    best_score = -1

    for match, score, _ in candidates:
        length_score = 1 - abs(len(match) - len(word)) / max(len(match), len(word))

        first_letter_bonus = 1 if match[0] == word[0] else 0

        final_score = (
            score * 0.7 +
            length_score * 20 +
            first_letter_bonus * 10
        )

        if final_score > best_score:
            best_score = final_score
            best_match = match

    return best_match, best_score

In [45]:
def predict_full_pipeline(image_path):
    # OCR
    raw = predict_image(image_path)

    # correction + confidence
    corrected, confidence = correct_text_with_confidence(raw)

    # generic mapping
    generic = name_map.get(corrected, "unknown")

    return {
        "ocr": raw,
        "corrected": corrected,
        "generic": generic,
        "confidence": round(confidence, 2)
    }

In [46]:
result = predict_full_pipeline("150.png")

print(result)

{'ocr': 'bicozin', 'corrected': 'bicozin', 'generic': 'vitamin b complex + zinc', 'confidence': 100.0}


In [ ]:
import json

with open("/content/drive/MyDrive/drug_list.json", "w") as f:
    json.dump(drug_list, f)

In [ ]:
with open("/content/drive/MyDrive/name_map.json", "w") as f:
    json.dump(name_map, f)

In [217]:
with open("predict.py", "w") as f:
    f.write("""
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms

# -------------------------
# VOCAB
# -------------------------
import string
vocab = list(string.ascii_lowercase)
vocab.append("<blank>")

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# -------------------------
# MODEL
# -------------------------
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
        )

        self.rnn = nn.LSTM(
            input_size=256 * 8,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()

        x = x.permute(0, 3, 1, 2)
        x = x.view(b, w, c * h)

        x, _ = self.rnn(x)
        x = self.fc(x)

        return x

# -------------------------
# TRANSFORM
# -------------------------
transform = transforms.Compose([
    transforms.Resize((32, 256)),
    transforms.ToTensor()
])

# -------------------------
# DECODE
# -------------------------
def decode_predictions(outputs, idx2char, blank):
    preds = outputs.argmax(2)
    preds = preds.permute(1, 0)

    results = []
    for pred in preds:
        string = ""
        prev = -1
        for p in pred:
            p = p.item()
            if p != prev and p != blank:
                string += idx2char[p]
            prev = p
        results.append(string)
    return results

# -------------------------
# LOAD MODEL
# -------------------------
model = CRNN(num_classes=len(vocab))
model.load_state_dict(torch.load("crnn_prescription.pth", map_location="cpu"))
model.eval()

# -------------------------
# PREDICT FUNCTION
# -------------------------
def predict_image(image_path):
    img = Image.open(image_path).convert("L")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img)
        outputs = outputs.log_softmax(2)

    pred = decode_predictions(outputs, idx2char, char2idx["<blank>"])
    return pred[0]

# -------------------------
# MAIN
# -------------------------
if __name__ == "__main__":
    print(predict_image("150.png"))
""")